# Phase 6: Model Evaluation and Selection
In this notebook, we load the trained models from Phase 5 and evaluate them on unseen testing data. We generate classification reports, confusion matrices, and cross-validation scores to confidently select the most robust model for deployment.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score

plt.style.use('ggplot')

## 1. Load Models & Data
We load the serialized TF-IDF testing sparse matrices, the target arrays, the `LabelEncoder`, and our three predictive models.

In [ ]:
X_train = joblib.load('../models/X_train.pkl')
y_train = joblib.load('../models/y_train.pkl')
X_test = joblib.load('../models/X_test.pkl')
y_test = joblib.load('../models/y_test.pkl')
label_encoder = joblib.load('../models/label_encoder.pkl')

dt_model = joblib.load('../models/decision_tree_model.pkl')
knn_model = joblib.load('../models/knn_model.pkl')
svm_model = joblib.load('../models/svm_model.pkl')

class_names = label_encoder.classes_

models = {
    'Decision Tree': dt_model,
    'KNN': knn_model,
    'SVM': svm_model
}

print("All models and data loaded successfully.")

## 2. Evaluate Models (Predictions & Metrics)
Iterating through each model to generate predictions on `X_test`. We extract Accuracy, Precision, Recall, and F1-Scores.

In [ ]:
metrics_data = []

for name, model in models.items():
    print(f"\n{'='*40}")
    print(f"Evaluating: {name}")
    print(f"{'='*40}")
    
    # Generate Predictions
    preds = model.predict(X_test)
    
    # Evaluation Metrics
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    
    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=class_names, zero_division=0))
    
    # Cross Validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    cv_mean = cv_scores.mean()
    print(f"5-Fold CV Mean Accuracy: {cv_mean:.4f} (+/- {cv_scores.std():.4f})")
    
    metrics_data.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'Cross Validation Mean': cv_mean
    })

## 3. Model Comparison Table
Consolidating all extracted metrics into a single Pandas DataFrame.

In [ ]:
df_metrics = pd.DataFrame(metrics_data)
print("=== Model Comparison Table ===")
display(df_metrics)

## 4. Visualizing Performance
Plotting bar charts to compare Accuracy, Precision, Recall, F1, and Cross Validation Mean across our algorithms.

In [ ]:
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'Cross Validation Mean']

for metric in metrics_to_plot:
    plt.figure(figsize=(8, 5))
    ax = sns.barplot(x='Model', y=metric, data=df_metrics, hue='Model', palette='viridis', legend=False)
    plt.title(f'{metric} Comparison')
    plt.ylim(0, 1.1)
    for i, v in enumerate(df_metrics[metric]):
        plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
    plt.tight_layout()
    plt.show()

## 5. Model Selection & Final Output
Automatically computing the best model based on weighted F1 score and Cross-Validation stability. The best model will be persisted as `models/career_model.pkl` for production use.

In [ ]:
# Determine the best model automatically
best_model_row = df_metrics.sort_values(by=['Cross Validation Mean', 'Accuracy', 'F1'], ascending=[False, False, False]).iloc[0]
best_model_name = best_model_row['Model']

print(f"Best Model Selected: {best_model_name}")

if best_model_name == 'Decision Tree':
    shutil.copy('../models/decision_tree_model.pkl', '../models/career_model.pkl')
elif best_model_name == 'KNN':
    shutil.copy('../models/knn_model.pkl', '../models/career_model.pkl')
else:
    shutil.copy('../models/svm_model.pkl', '../models/career_model.pkl')

print(f"Verified models/career_model.pkl exists: {os.path.exists('../models/career_model.pkl')}")